# Module 01 — How Spark Works

Instructor template. Open Spark UI before running actions.

In [ ]:
import sys
sys.path.insert(0, '/workspace' if '/workspace' in sys.path[0] or __import__('pathlib').Path('/workspace').exists() else '.')

from shared.spark_session import get_spark, ui_links, explain_plan
from shared import config

spark = get_spark('module-01-how-spark-works')
ui_links()

## Load Data

NYC yellow taxi trips + zone lookup (public TLC dataset).

In [ ]:
trips = spark.read.parquet(*config.yellow_taxi_paths())
zones = spark.read.option('header', True).csv(config.zones_path())
print(f'Trips: {trips.count():,} rows')
print(f'Zones: {zones.count():,} rows')

## 1. Aggregation

Run a `groupBy` + `count`. Open **Jobs** and **Stages** tabs at http://localhost:4040

In [ ]:
vendor_counts = trips.groupBy('VendorID').count().orderBy('count', ascending=False)
vendor_counts.show()
# Observe: 1 job, typically 2 stages (partial agg + final agg or shuffle)

In [ ]:
explain_plan(vendor_counts)

## 2. Window Function

Window functions often trigger a shuffle to distribute rows within each partition group.

In [ ]:
from pyspark.sql import Window
from pyspark.sql.functions import row_number, col

w = Window.partitionBy('VendorID').orderBy(col('tpep_pickup_datetime').desc())
ranked = trips.withColumn('rn', row_number().over(w)).filter('rn <= 3')
ranked.select('VendorID', 'tpep_pickup_datetime', 'rn').show(10)
# Observe: Exchange (shuffle) stage in UI

In [ ]:
explain_plan(ranked)

## 3. Join

Join trips to zone lookup on `PULocationID = LocationID`. Watch shuffle read/write in Stages tab.

In [ ]:
joined = trips.join(
    zones.withColumnRenamed('LocationID', 'PULocationID'),
    on='PULocationID',
    how='left'
)
joined.select('PULocationID', 'Zone', 'Borough', 'total_amount').show(5)
# Observe: BroadcastExchange (small zones table) or SortMergeJoin + Exchange

In [ ]:
explain_plan(joined)

## 4. Spark UI Literacy

| UI Level | What it represents |
|----------|--------------------|
| **Job** | One action triggered this job |
| **Stage** | Pipeline between shuffles |
| **Task** | One partition's work on one executor |

Check **SQL** tab for the physical plan linked to each query.

## 5. Query Planner Non-Determinism

The same logical query can produce different physical plans depending on stats and configs.

In [ ]:
# Plan A: default broadcast threshold
spark.conf.set('spark.sql.autoBroadcastJoinThreshold', '10MB')
small = zones.limit(100)
plan_a = trips.join(small.withColumnRenamed('LocationID', 'PULocationID'), 'PULocationID')
plan_a.explain('formatted')

In [ ]:
# Plan B: disable broadcast — forces shuffle join
spark.conf.set('spark.sql.autoBroadcastJoinThreshold', '-1')
plan_b = trips.join(zones.withColumnRenamed('LocationID', 'PULocationID'), 'PULocationID')
plan_b.explain('formatted')
# Compare: BroadcastHashJoin vs SortMergeJoin

In [ ]:
# Reset
spark.conf.set('spark.sql.autoBroadcastJoinThreshold', '10MB')

## Key Takeaways

1. Transformations are lazy; actions trigger jobs.
2. Shuffles (Exchange) create new stages — expensive in distributed systems.
3. Catalyst chooses plans based on data size, stats, and configs — not fixed by your code alone.